# Lecture 2.1 — Agent Anatomy: name, instructions, model, tools, handoffs, guardrails

**Section 02 — Agents: Configuration & Behaviour**

In this notebook we map every property the `Agent` dataclass accepts. By the end you will know what each property does, which ones we use today, and which lectures will cover the rest in depth.

---

## Cell 1 — Install the OpenAI Agents SDK

We pin to the latest release of `openai-agents`. If you already ran this cell in a previous lecture during this Colab session, pip will confirm the package is already present and move on instantly — no harm in re-running it.

> **Why this cell exists in every notebook:** Each notebook in this course is self-contained. A learner opening this notebook for the first time needs this cell to run before any other cell will work.

In [1]:
!pip install openai-agents -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.8/850.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.7 MB/s eta 0:00:00


## Cell 2 — API Key Setup (Google Colab Secrets)

We retrieve the OpenAI API key from Colab's built-in Secrets store and write it to the `OPENAI_API_KEY` environment variable. The SDK reads this variable automatically — you never need to pass the key explicitly to `Agent` or `Runner`.

### How to add your key in Colab

1. Click the **🔑 key icon** in the left sidebar ("Secrets").
2. Click **"+ Add new secret"**.
3. Set **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the **Value**.
5. Toggle **"Notebook access"** to ON for this notebook.
6. Run the cell below.

> **Running locally?** Skip this cell and instead set the environment variable in your terminal before launching Jupyter:
> ```bash
> export OPENAI_API_KEY="sk-..."
> ```

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Imports

We import the three core objects we need for this lecture:

| Import | What it is |
|--------|------------|
| `Agent` | The dataclass that defines an agent's identity, behaviour, and capabilities. Every property we explore in this lecture lives here. |
| `ModelSettings` | A companion dataclass for model-level tuning parameters (temperature, reasoning effort, verbosity, etc.). Introduced briefly in this lecture; covered in depth in Lecture 2.2. |
| `Runner` | The execution engine that runs an agent turn-by-turn. We use `Runner.run()` exclusively in all notebooks — see Rule F4. |

> **Note:** `asyncio` is not imported here because Colab and Jupyter already have a running event loop — `await` works at the top level of any cell without any additional setup.

In [3]:
from agents import Agent, ModelSettings, Runner

## Cell 4 — Full Agent Property Reference

Before writing any code, let's read the full map of what `Agent` accepts. This table is the conceptual anchor for the entire lecture — and for the whole of Section 2.

Every property listed here is verified against `src/agents/agent.py` in the SDK source.

| Property | Required | Type | Description |
|----------|----------|------|-------------|
| `name` | **Yes** | `str` | Human-readable agent name. Shows up in traces, handoff routing, and the agent visualiser. |
| `instructions` | No | `str \| Callable \| None` | The system prompt. Can be a static string or a dynamic callback. Strongly recommended. |
| `model` | No | `str \| Model \| None` | Which LLM to use. When not set, the SDK uses `gpt-5.4-mini` with `reasoning.effort="none"` and `verbosity="low"`. |
| `model_settings` | No | `ModelSettings` | Tuning parameters: temperature, top_p, reasoning effort, verbosity, tool_choice, etc. |
| `tools` | No | `list` | Tools the agent can call (function tools, hosted tools, agents-as-tools). |
| `handoffs` | No | `list` | Sub-agents the agent can delegate the conversation to. |
| `input_guardrails` | No | `list` | Checks that run on the first user input before the agent processes it. |
| `output_guardrails` | No | `list` | Checks that run on the agent's final output before delivery. |
| `output_type` | No | `type \| AgentOutputSchemaBase \| None` | Structured output type instead of plain text. Defaults to `str`. |
| `hooks` | No | `AgentHooks \| None` | Agent-scoped lifecycle callbacks (on_start, on_tool_call, on_end, etc.). |
| `tool_use_behavior` | No | `"run_llm_again" \| "stop_on_first_tool" \| StopAtTools \| ToolsToFinalOutputFunction` | Controls whether tool results loop back to the model or end the run immediately. Default: `"run_llm_again"`. |
| `reset_tool_choice` | No | `bool` | Resets `tool_choice` after a tool call to prevent infinite tool loops. Default: `True`. |
| `handoff_description` | No | `str \| None` | Short description shown when this agent is offered as a handoff target to another agent. |
| `mcp_servers` | No | `list` | MCP-backed tool servers for the agent. |

---

In this lecture we will build live examples for `name`, `instructions`, `model`, and `model_settings`. Every other property has its own dedicated lecture or section — see the Course Map at the end of this notebook.

## Cell 5 — Minimal Agent: Just a Name

The only **required** property on `Agent` is `name`. Everything else has a sensible default.

What happens when we run an agent with no `instructions`?

- The SDK uses `gpt-5.4-mini` as the model (the default).
- The model receives **no system prompt** — it falls back to its own default behaviour.
- The agent still runs and produces output, but it has no defined persona, focus, or constraints.

This cell demonstrates that the minimal viable agent is one line — and shows you exactly what you give up by omitting `instructions`.

**Teaching point:** The contrast with Cell 6 (which adds `instructions`) makes the impact of the system prompt immediately visible.

In [4]:
agent_minimal = Agent(name="Minimal")

result = await Runner.run(agent_minimal, "Say hello.")
print(result.final_output)

Hello!


## Cell 6 — Adding Instructions: The System Prompt

The `instructions` property is the system prompt for the agent. It tells the model:

- **Who it is** — its persona
- **What it does** — its focus and purpose
- **How it responds** — its style, format, and tone

The SDK docs describe `instructions` as *strongly recommended* — and mean it. An agent without instructions is an LLM with no persona, no focus, and no guardrails on its behaviour.

Here we give the agent a clear persona — a poet who always responds in rhyming couplets — and ask it the same question as the minimal agent. The contrast in output makes the difference unmistakable.

**Note on multi-line strings:** We use Python's implicit string concatenation inside parentheses to keep the instructions readable. This is equivalent to a single string with a space in the middle.

In [5]:
agent_with_instructions = Agent(
    name="Poet",
    instructions=(
        "You are a poet. "
        "Always respond in rhyming couplets."
    ),
)

result = await Runner.run(
    agent_with_instructions,
    "Explain what an API is.",
)
print(result.final_output)

An API is a way for apps to talk and share,  
A set of rules and doors that show what’s there.  

It’s like a menu: you ask, it brings the dish,  
Without knowing how it’s made—just what you wish.


## Cell 7 — Specifying the Model Explicitly

The `model` property accepts a **string model name**. When you do not set it, the SDK defaults to `gpt-5.4-mini`.

Here we pass `"gpt-5.4-mini"` explicitly — this produces the **same behaviour as omitting `model` entirely**, but makes the choice visible in code. This is useful for:

- Making your code self-documenting (the reader knows which model is being used)
- Pinning to a specific model so your app is not affected by SDK default changes in the future

**Key facts about the default model:**

| Setting | Value |
|---------|-------|
| Default model | `gpt-5.4-mini` |
| Auto-applied `reasoning.effort` | `"none"` |
| Auto-applied `verbosity` | `"low"` |
| Optimised for | Low-latency agent workflows |

> **Coming up:** Lecture 2.2 is a full deep-dive into model selection — including all available GPT-5 variants, how to switch models, and how to override defaults via `RunConfig`.

In [6]:
agent_with_model = Agent(
    name="Nano Poet",
    instructions=(
        "You are a poet. "
        "Always respond in rhyming couplets."
    ),
    model="gpt-5.4-mini",
)

result = await Runner.run(
    agent_with_model,
    "Explain what an API is.",
)
print(result.final_output)

An API is a bridge that lets two systems speak,  
A set of rules and doors so they can meet and seek.  

One program asks, another answers, neat and clean,  
Like ordering at a counter, with a service in between.


## Cell 8 — Adding ModelSettings: A Brief Introduction

`ModelSettings` is where all model-level tuning parameters live. In this cell we introduce the pattern — the full deep-dive is Lecture 2.2.

We switch to `gpt-5.5` (the higher-quality GPT-5 model) and explicitly set two parameters:

| Parameter | Value | What it does |
|-----------|-------|--------------|
| `reasoning=Reasoning(effort="low")` | `"low"` | Controls how much internal reasoning the model does before responding. GPT-5 models require this setting. |
| `verbosity` | `"low"` | Controls the verbosity of the model's reasoning trace. `"low"` keeps the output concise. |

**Important import note:** `Reasoning` is imported from `openai.types.shared` — not from the `agents` package. This is because `Reasoning` is a type defined by the OpenAI Python SDK, which the Agents SDK builds on top of.

**Why explicit settings when the SDK has defaults?**
When you pass a GPT-5 model string, the SDK automatically applies sensible `ModelSettings` defaults. However, being explicit in your code:
1. Makes your intent clear to readers
2. Gives you control if defaults change in a future SDK version
3. Builds the habit of always thinking about reasoning cost and latency

> **Coming up:** Lecture 2.2 covers the full `ModelSettings` surface: `temperature`, `top_p`, `max_tokens`, `tool_choice`, all reasoning effort levels, and how to apply settings at the `RunConfig` level.

In [9]:
from openai.types.shared import Reasoning

agent_with_settings = Agent(
    name="Creative Poet",
    instructions=(
        "You are a poet. "
        "Always respond in rhyming couplets."
    ),
    model="gpt-5.5",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="low"), # How hard the model thinks?
        verbosity="low", # How wordy the final output is?
    ),
)

result = await Runner.run(
    agent_with_settings,
    "Explain what an API is.",
)
print(result.final_output)

An API is a doorway apps can use,  
To ask for data, services, or news.  

It sets the rules for how requests are made,  
And how replies come back, neatly displayed.  

Like ordering food from a menu’s list,  
You ask, it serves—no kitchen details missed.


## Course Map — Where Every Property Is Taught

We have now seen `name`, `instructions`, `model`, and `model_settings` in action. Every remaining `Agent` property gets its own dedicated lecture or section. Use this table as your roadmap.

| Property | Covered in |
|----------|------------|
| `tools` | Section 3 |
| `handoffs` | Lecture 5.2 |
| `input_guardrails` | Lecture 5.8 |
| `output_guardrails` | Lecture 5.9 |
| `output_type` | Lecture 2.5 |
| `hooks` | Lecture 6.4 |
| `tool_use_behavior` | Lecture 2.8 |
| `reset_tool_choice` | Lecture 2.7 |
| `handoff_description` | Lecture 5.2 |
| `mcp_servers` | Update Section U2 |

Every single one of these properties will get its own lecture or section. By the end of Section 5 you will have used all of them.